# 🔍 02 - AI-Powered Action Extraction

**COMPLETELY INDEPENDENT NOTEBOOK** - Run this after notebook 01.

## What this notebook does:
- ✅ Loads data from `quickstart_catalog_vkm_external.classify_tickets.raw_tickets`  
- ✅ Uses **Databricks AI Functions** for intelligent extraction
- ✅ Extracts actions, priorities, and timelines using LLMs
- ✅ Categorizes action items by type  
- ✅ Saves results to Unity Catalog tables  
- ✅ Ready for next notebook: `03_ai_classification.ipynb`

**Prerequisites:** Run `01_sample_data_generation.ipynb` first

## AI Functions Used:
- `ai_classify` - For priority and category classification
- `ai_extract` - For action items extraction  
- `ai_gen` - For complex analysis and summaries


In [1]:
# Import required libraries
import pandas as pd
import re
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")


✅ Libraries imported and configuration loaded


In [2]:
# Load Sample Data
print("📊 Loading sample ticket data from Unity Catalog...")

# Load the sample ticket data from Unity Catalog
df_tickets = spark.table(TABLES["raw_tickets"])
print(f"✅ Loaded {df_tickets.count()} tickets from: {TABLES['raw_tickets']}")

# Display sample data
print("\n📋 Sample data:")
display(df_tickets.select("ticket_id", "short_description", "description"))


📊 Loading sample ticket data from Unity Catalog...


✅ Loaded 10 tickets from: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📋 Sample data:


,ticket_id,short_description,description
0,TICKET_001,Issue #1 - Critical,hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.
1,TICKET_002,Issue #2 - Problem,urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.
2,TICKET_003,Issue #3 - Urgent,i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?
3,TICKET_004,Issue #4 - Important,our monitoring alerts are going crazy. everything shows red but the app seems to be working fine. not sure if it's a false positive or if something is actually broken. help?
4,TICKET_005,Issue #5 - Problem,the deployment failed again and now production is down. we rolled back but need to figure out what went wrong. this is the third time this month. we need better testing.
5,TICKET_006,Issue #6 - Critical,customers are reporting that their data is missing after the last update. this is a big problem and we need to investigate immediately. could be a data migration issue.
6,TICKET_007,Issue #7 - Problem,the new security patch broke our authentication. users can't log in and we're getting flooded with support tickets. need to fix this before the security team gets involved.
7,TICKET_008,Issue #8 - Problem,our cloud costs are through the roof this month. something is using way more resources than usual. need to find out what's causing the spike and optimize it.
8,TICKET_009,Issue #9 - Problem,the backup system isn't working and we haven't had a successful backup in 3 days. this is a major risk and we need to fix it before something bad happens.
9,TICKET_010,Issue #10 - Important,the new microservice is causing memory leaks and crashing the whole system. we need to either fix it or disable it until we can figure out what's wrong.


In [3]:
# AI-Powered Extraction Using Databricks AI Functions
print("🤖 Step 1: Classifying ticket priorities using ai_classify...")

# 1. PRIORITY CLASSIFICATION using ai_classify (SQL syntax)
df_with_priority = df_tickets.selectExpr(
    "*",
    "ai_classify(description, ARRAY('Low Priority', 'Medium Priority', 'High Priority', 'Urgent Priority')) as priority_classified"
)

print("✅ Priority classification completed")
display(df_with_priority.select("ticket_id", "short_description", "priority_classified"))


🤖 Step 1: Classifying ticket priorities using ai_classify...
✅ Priority classification completed


,ticket_id,short_description,priority_classified
0,TICKET_001,Issue #1 - Critical,Urgent Priority
1,TICKET_002,Issue #2 - Problem,Urgent Priority
2,TICKET_003,Issue #3 - Urgent,Medium Priority
3,TICKET_004,Issue #4 - Important,High Priority
4,TICKET_005,Issue #5 - Problem,High Priority
5,TICKET_006,Issue #6 - Critical,Urgent Priority
6,TICKET_007,Issue #7 - Problem,Urgent Priority
7,TICKET_008,Issue #8 - Problem,High Priority
8,TICKET_009,Issue #9 - Problem,Urgent Priority
9,TICKET_010,Issue #10 - Important,Urgent Priority


In [4]:
# 2. ACTION ITEMS EXTRACTION using ai_gen (better for arrays)
print("🔍 Step 2: Extracting action items using ai_gen...")

df_with_actions = df_with_priority.selectExpr(
    "*",
    "ai_gen(CONCAT('Extract all specific action items, tasks, requirements, and deliverables from this ticket description. Return as a JSON array of strings. Each item should be a concrete, actionable task. If no action items are found, return an empty array []. Ticket description: ', description)) as action_items_raw"
)

# Debug: Let's see what the raw output looks like
print("🔍 Debug: Raw AI output for first ticket:")
df_with_actions.select("ticket_id", "action_items_raw").show(1, truncate=False)

# Parse the JSON response and convert to array
df_with_actions_parsed = df_with_actions.withColumn(
    "action_items_extracted",
    from_json(col("action_items_raw"), ArrayType(StringType()))
).withColumn(
    "action_items_extracted",
    when(col("action_items_extracted").isNull(), array()).otherwise(col("action_items_extracted"))
).drop("action_items_raw")

print("✅ Action items extraction completed")
display(df_with_actions_parsed.select("ticket_id", "short_description", "action_items_extracted"))


🔍 Step 2: Extracting action items using ai_gen...
🔍 Debug: Raw AI output for first ticket:


+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

,ticket_id,short_description,action_items_extracted
0,TICKET_001,Issue #1 - Critical,[]
1,TICKET_002,Issue #2 - Problem,[]
2,TICKET_003,Issue #3 - Urgent,[]
3,TICKET_004,Issue #4 - Important,[]
4,TICKET_005,Issue #5 - Problem,[]
5,TICKET_006,Issue #6 - Critical,[]
6,TICKET_007,Issue #7 - Problem,[]
7,TICKET_008,Issue #8 - Problem,[]
8,TICKET_009,Issue #9 - Problem,[]
9,TICKET_010,Issue #10 - Important,[]


In [5]:
# 3. MAIN ACTION REQUEST using ai_gen (SQL syntax)
print("🎯 Step 3: Extracting main action request using ai_gen...")

df_with_main_action = df_with_actions_parsed.selectExpr(
    "*",
    "ai_gen(CONCAT('Extract the main action or request from this ticket description. Return only the specific action requested, not a list of items. If no clear action is requested, return None. Ticket description: ', description)) as action_requested"
)

print("✅ Main action request extraction completed")
display(df_with_main_action.select("ticket_id", "short_description", "action_requested"))

# 4. TIMELINE EXTRACTION using ai_gen (SQL syntax)
print("⏰ Step 4: Extracting timeline information using ai_gen...")

df_with_timeline = df_with_main_action.selectExpr(
    "*",
    "ai_gen(CONCAT('Extract timeline information from this ticket description. Look for due dates, deadlines, urgency indicators, or time references. Return the specific timeline mentioned, or None if no timeline is specified. Ticket description: ', description)) as timeline_extracted"
)

print("✅ Timeline extraction completed")
display(df_with_timeline.select("ticket_id", "short_description", "timeline_extracted"))


🎯 Step 3: Extracting main action request using ai_gen...
✅ Main action request extraction completed


,ticket_id,short_description,action_requested
0,TICKET_001,Issue #1 - Critical,Investigate and resolve the issue with the website's slow performance.
1,TICKET_002,Issue #2 - Problem,Fix the login system.
2,TICKET_003,Issue #3 - Urgent,Debug the API errors.
3,TICKET_004,Issue #4 - Important,Investigate the monitoring alerts to determine if they are false positives or indicative of an actual issue.
4,TICKET_005,Issue #5 - Problem,Investigate the cause of the deployment failure.
5,TICKET_006,Issue #6 - Critical,"Investigate the missing customer data issue, potentially related to a data migration problem."
6,TICKET_007,Issue #7 - Problem,Fix the authentication issue caused by the new security patch.
7,TICKET_008,Issue #8 - Problem,Find out what's causing the spike in cloud costs and optimize it.
8,TICKET_009,Issue #9 - Problem,Fix the backup system.
9,TICKET_010,Issue #10 - Important,Fix or disable the new microservice to prevent memory leaks and system crashes.


⏰ Step 4: Extracting timeline information using ai_gen...
✅ Timeline extraction completed


,ticket_id,short_description,timeline_extracted
0,TICKET_001,Issue #1 - Critical,"The timeline mentioned in the ticket description is: ""since this morning"". This indicates that the issue started at some point this morning, but no specific due date, deadline, or urgency indicator (like ""ASAP"" or ""urgent"") is mentioned beyond the fact that sales are being lost, implying a need for prompt attention."
1,TICKET_002,Issue #2 - Problem,"The timeline information mentioned in the ticket description is:\n\n* ""ASAP"" (as soon as possible), indicating a high level of urgency\n* ""last week"", referencing a previous incident, but not providing a specific deadline or due date for the current issue.\n\nNo specific due date or deadline is mentioned, but the urgency is high."
2,TICKET_003,Issue #3 - Urgent,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The request is for general assistance with debugging an issue, but it does not include any time-sensitive information."
3,TICKET_004,Issue #4 - Important,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The description is a general request for help with an issue, but it does not include any time-related information."
4,TICKET_005,Issue #5 - Problem,"The specific timeline mentioned is: ""this month"". This indicates that the issue has occurred three times within the current month, but it does not provide a specific due date or deadline."
5,TICKET_006,Issue #6 - Critical,"The timeline information mentioned in the ticket description is: ""immediately"". This indicates a sense of urgency, but does not provide a specific due date or deadline. There is also a reference to ""the last update"", which implies that the issue occurred recently, but the exact time frame is not specified. \n\nSo, the extracted timeline information is: ""immediately"" (indicating high urgency, but no specific date or time frame)."
6,TICKET_007,Issue #7 - Problem,"The timeline information mentioned in the ticket description is: ""before the security team gets involved"". This implies a sense of urgency, but does not specify a particular due date or deadline. However, it can be inferred that the issue needs to be resolved as soon as possible to avoid escalation to the security team. \n\nSince there is no specific date or time mentioned, the extracted timeline information is somewhat vague, but it does indicate a need for prompt action."
7,TICKET_008,Issue #8 - Problem,"None \n\nThere is no specific due date, deadline, urgency indicator, or time reference mentioned in the ticket description, apart from the general mention of ""this month"", which is not a specific timeline for the task itself."
8,TICKET_009,Issue #9 - Problem,"The timeline information mentioned in the ticket description is:\n\n* 3 days (the time since the last successful backup)\n\nThere is also an implied urgency to fix the issue as soon as possible to prevent potential problems, but no specific due date or deadline is mentioned."
9,TICKET_010,Issue #10 - Important,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The description only mentions the need to fix or disable the microservice, but does not provide any time-related information."


In [6]:
# Action Items Categorization
print("🏷️ Step 5: Categorizing action items...")

# Explode action items to create one row per action item
df_action_items_exploded = df_with_timeline.select(
    "ticket_id",
    "short_description", 
    "action_requested",
    "priority_classified",
    "timeline_extracted",
    explode("action_items_extracted").alias("action_item")
).filter(col("action_item").isNotNull())

print(f"✅ Exploded {df_action_items_exploded.count()} individual action items")

# Categorize action items using ai_classify (SQL syntax)
df_action_items_categorized = df_action_items_exploded.selectExpr(
    "*",
    "ai_classify(action_item, ARRAY('Infrastructure', 'Security', 'Monitoring', 'Testing', 'Documentation', 'Network', 'Database', 'Application', 'Other')) as action_category"
)

print("✅ Action items categorization completed")
display(df_action_items_categorized)


🏷️ Step 5: Categorizing action items...


✅ Exploded 0 individual action items
✅ Action items categorization completed


,ticket_id,short_description,action_requested,priority_classified,timeline_extracted,action_item,action_category


In [7]:
# Save Extracted Data
print("💾 Step 6: Saving extracted data to Unity Catalog...")

try:
    # Save the main extracted data to Unity Catalog
    df_with_timeline.write.format("delta").mode("overwrite").saveAsTable(TABLES["tickets_with_actions"])
    print("✅ Main data saved successfully!")
    
    # Save the categorized action items to Unity Catalog  
    df_action_items_categorized.write.format("delta").mode("overwrite").saveAsTable(TABLES["action_items_detailed"])
    print("✅ Action items saved successfully!")
    
    print("✅ AI-extracted data saved to Unity Catalog Delta tables:")
    print(f"📊 Main data: {TABLES['tickets_with_actions']}")
    print(f"📋 Action items: {TABLES['action_items_detailed']}")
    
    # Show summary statistics
    print(f"\n📈 Summary Statistics:")
    print(f"   Total tickets processed: {df_with_timeline.count()}")
    print(f"   Total action items extracted: {df_action_items_categorized.count()}")
    
    # Show action category distribution
    print(f"\n📊 Action Category Distribution:")
    df_action_items_categorized.groupBy("action_category").count().orderBy(desc("count")).show()
    
    # Show priority distribution
    print(f"\n🎯 Priority Distribution:")
    df_with_timeline.groupBy("priority_classified").count().orderBy(desc("count")).show()
    
    print("\n🎯 Ready for next notebook: 03_ai_classification.ipynb")
    
except Exception as e:
    print(f"❌ Error saving data: {e}")
    print("🔍 Trying to save to temporary tables...")
    try:
        df_with_timeline.write.format("delta").mode("overwrite").saveAsTable("tickets_with_actions_temp")
        df_action_items_categorized.write.format("delta").mode("overwrite").saveAsTable("action_items_detailed_temp")
        print("✅ Saved to temporary tables")
    except Exception as e2:
        print(f"❌ Error with temp tables: {e2}")


💾 Step 6: Saving extracted data to Unity Catalog...


✅ Main data saved successfully!


✅ Action items saved successfully!
✅ AI-extracted data saved to Unity Catalog Delta tables:
📊 Main data: quickstart_catalog_vkm_external.classify_tickets.tickets_with_actions
📋 Action items: quickstart_catalog_vkm_external.classify_tickets.action_items_detailed

📈 Summary Statistics:


   Total tickets processed: 10


   Total action items extracted: 0

📊 Action Category Distribution:


+---------------+-----+
|action_category|count|
+---------------+-----+
+---------------+-----+


🎯 Priority Distribution:


+-------------------+-----+
|priority_classified|count|
+-------------------+-----+
|    Urgent Priority|    6|
|      High Priority|    3|
|    Medium Priority|    1|
+-------------------+-----+


🎯 Ready for next notebook: 03_ai_classification.ipynb
